# Regionalizacion Nacional a Regional - Modelo OSeMOSYS SAND



Este notebook regionaliza parametros del **SAND Nacional** usando el **Regional existente como inventario de referencia** (regional-first).



Procesa los **8 parametros** de la seccion 5.3 para cada sector con porcentajes en [`Configuracion.xlsx`](Configuracion.xlsx): **IND, TRA, RES, TER, AGF**.



| Entrada | Descripcion |

|---------|-------------|

| **Nacional** | `01-04-2026 SAND BASE v10 - copia.xlsx` (BASE) o `scenario_CN_Nacional_166_Parameters_SAND.xlsx` (CN) |

| **Configuracion** | `Configuracion.xlsx` — hojas `Cap_*` / `Dem_*` por sector |

| **Regional existente** | `Escenario_SAND_Regional.xlsx` (inventario + comparacion) |



**Regiones:** `AN`, `SE`, `IN`, `NE`, `CA`, `OR`, `SO`



**Horizonte:** 2022-2054 (2055 excluido)



**Porcentajes en Config (solo 2 parametros aditivos):**

- `Cap_Ind`, `Cap_TRA`, `Cap_RES`, `Cap_TER`, `Cap_AGF` → `ResidualCapacity`

- `Dem_Ind`, `Dem_TRA`, `Dem_RES`, `Dem_TER`, `Dem_AGF` → `AccumulatedAnnualDemand`



**Parametros (seccion 5.3):**

- **Aditivos fijos:** `AccumulatedAnnualDemand`, `ResidualCapacity`

- **No aditivos fijos:** `CapitalCost`, `InputActivityRatio`, `OutputActivityRatio`

- **Condicionales:** limites de inversion/actividad (regla 9999/0)



**Salidas por sector:** `output/{ESCENARIO}_sand_regionalizado_{SECTOR}.xlsx` y `output/{ESCENARIO}_diferencias_regionalizacion_{SECTOR}.xlsx` (p. ej. `BASE_sand_regionalizado_TRA.xlsx`)



In [1]:
# --- 1. Configuracion general (multi-sector) ---

from __future__ import annotations



from collections import defaultdict

from dataclasses import dataclass

from pathlib import Path

from typing import Any, Callable



import numpy as np

import pandas as pd

from IPython.display import Markdown, display



BASE_DIR = Path(".").resolve()

OUTPUT_DIR = BASE_DIR / "output"

CONFIG_FILE = BASE_DIR / "Configuracion.xlsx"

REGIONAL_FILE = BASE_DIR / "Regional" / "Escenario_SAND_Regional.xlsx"

SHEET_NAME = "Parameters"

TIME_INDEP_COL = "Time indipendent variables"

ANIO_MAX = 2054

DIMENSION_COLS = [

    "Parameter", "REGION", "TECHNOLOGY", "EMISSION", "MODE_OF_OPERATION",

    "FUEL", "TIMESLICE", "STORAGE", "REGION2",

]

REGIONES = ["AN", "SE", "IN", "NE", "CA", "OR", "SO"]

TOLERANCE = 0.01



# Escenario: "BASE" o "CN"

ESCENARIO = "BASE"

OUTPUT_PREFIX = ESCENARIO  # prefijo de archivos en output/ (BASE o CN)

NACIONAL_FILE = (

    BASE_DIR / "Nacional" / "01-04-2026 SAND BASE v10 - copia.xlsx"

    if ESCENARIO == "CN"

    else BASE_DIR / "Nacional" / "01-04-2026 SAND BASE v10 - copia.xlsx"

)



SECTORS_CONFIG: dict[str, dict[str, Any]] = {

    "IND": {

        "filter": lambda tech, fuel: "IND" in str(tech) or "IND" in str(fuel),

        "hoja_cap": "Cap_Ind",

        "hoja_dem": "Dem_Ind",

    },

    "TRA": {

        "filter": lambda tech, fuel: "TRA" in str(tech) or "TRA" in str(fuel),

        "hoja_cap": "Cap_TRA",

        "hoja_dem": "Dem_TRA",

    },

    "RES": {

        "filter": lambda tech, fuel: "RES" in str(tech) or "RES" in str(fuel),

        "hoja_cap": "Cap_RES",

        "hoja_dem": "Dem_RES",

    },

    "TER": {

        "filter": lambda tech, fuel: "TER" in str(tech) or "TER" in str(fuel),

        "hoja_cap": "Cap_TER",

        "hoja_dem": "Dem_TER",

    },

    "AGF": {

        "filter": lambda tech, fuel: "AGF" in str(tech) or "AGF" in str(fuel),

        "hoja_cap": "Cap_AGF",

        "hoja_dem": "Dem_AGF",

    },

}



SECTORES_A_EJECUTAR = list(SECTORS_CONFIG.keys())



PARAMS_CONDICIONALES = {

    "TotalAnnualMaxCapacityInvestment",

    "TotalTechnologyAnnualActivityUpperLimit",

    "TotalTechnologyModelPeriodActivityUpperLimit",

}



PARAMS_CONFIG = {

    "AccumulatedAnnualDemand": {"dim": "fuel", "tipo": "aditivo", "dim_label": "FUEL", "pct_source": "dem"},

    "ResidualCapacity": {"dim": "tech", "tipo": "aditivo", "dim_label": "TECHNOLOGY", "pct_source": "cap"},

    "TotalAnnualMaxCapacityInvestment": {"dim": "tech", "tipo": "condicional", "dim_label": "TECHNOLOGY", "pct_source": None},

    "TotalTechnologyAnnualActivityUpperLimit": {"dim": "tech", "tipo": "condicional", "dim_label": "TECHNOLOGY", "pct_source": None},

    "TotalTechnologyModelPeriodActivityUpperLimit": {"dim": "tech", "tipo": "condicional", "dim_label": "TECHNOLOGY", "pct_source": None},

    "CapitalCost": {"dim": "tech", "tipo": "no_aditivo", "dim_label": "TECHNOLOGY", "pct_source": None},

    "InputActivityRatio": {"dim": "tech_fuel_mode", "tipo": "no_aditivo", "dim_label": "TECH+FUEL+MODE", "pct_source": None},

    "OutputActivityRatio": {"dim": "tech_fuel_mode", "tipo": "no_aditivo", "dim_label": "TECH+FUEL+MODE", "pct_source": None},

}



OUTPUT_DIR.mkdir(parents=True, exist_ok=True)





def get_value_cols(df: pd.DataFrame) -> list[str]:

    year_cols = [c for c in df.columns if isinstance(c, int) or (isinstance(c, str) and str(c).isdigit())]

    cols = [TIME_INDEP_COL] if TIME_INDEP_COL in df.columns else []

    return cols + [str(c) for c in year_cols]





def filtrar_anos_2054(year_cols: list[str]) -> list[str]:

    return [c for c in year_cols if c != TIME_INDEP_COL and int(c) <= ANIO_MAX]





def load_parameters(path: Path) -> pd.DataFrame:

    df = pd.read_excel(path, sheet_name=SHEET_NAME)

    value_cols = get_value_cols(df)

    for col in DIMENSION_COLS + value_cols:

        if col not in df.columns:

            raise ValueError(f"Columna faltante en {path}: {col}")

    return df





def normalize_text(value: Any) -> str | None:

    if value is None or (isinstance(value, float) and np.isnan(value)):

        return None

    if pd.isna(value):

        return None

    text = str(value).strip()

    return text if text else None





def split_regional_name(name: Any) -> tuple[str | None, str | None]:

    normalized = normalize_text(name)

    if normalized is None:

        return None, None

    parts = normalized.split("_", 1)

    if len(parts) == 2 and parts[0] in REGIONES:

        return parts[0], parts[1]

    return None, normalized





def safe_float(value: Any) -> float | None:

    if value is None or (isinstance(value, float) and np.isnan(value)):

        return None

    if pd.isna(value):

        return None

    try:

        return float(value)

    except (TypeError, ValueError):

        return None





def round3(value: Any) -> float | None:

    val = safe_float(value)

    if val is None:

        return None

    return round(val, 3)





def is_nines_or_zero(val: Any) -> bool:

    f = safe_float(val)

    if f is None:

        return True

    if f == 0.0:

        return True

    if f == int(f):

        digits = str(int(abs(f)))

        if digits and set(digits) == {"9"}:

            return True

    return False





def nacional_es_todo_cero(nac_row: pd.Series, cols: list[str]) -> bool:

    for col in cols:

        val = safe_float(nac_row[col])

        if val is None:

            continue

        if val != 0.0:

            return False

    return True





def values_match_3dec(actual: Any, expected: Any) -> bool:

    a = round3(actual)

    e = round3(expected)

    if a is None and e is None:

        return True

    if a is None or e is None:

        return False

    return a == e





def get_tipo_efectivo(param: str, nac_row: pd.Series, compare_cols: list[str]) -> str:

    cfg_tipo = PARAMS_CONFIG[param]["tipo"]

    if cfg_tipo == "aditivo":

        return "aditivo"

    if cfg_tipo == "condicional":

        for col in compare_cols:

            if not is_nines_or_zero(nac_row[col]):

                return "aditivo"

        return "no_aditivo"

    return "no_aditivo"





def show_df(df: pd.DataFrame, title: str = "", max_rows: int = 30):

    if title:

        display(Markdown(f"**{title}**"))

    if df.empty:

        display(Markdown("_Sin registros._"))

        return

    display(df.head(max_rows).style.hide(axis="index"))





def parse_pct_block(sheet_name: str) -> pd.DataFrame:

    raw = pd.read_excel(CONFIG_FILE, sheet_name=sheet_name, header=None)

    header_idx = None

    for i in range(min(20, len(raw))):

        row_vals = [normalize_text(v) for v in raw.iloc[i].tolist()]

        if row_vals and "Parametro" in row_vals:

            header_idx = i

            break

    if header_idx is None:

        raise ValueError(f"No se encontro encabezado 'Parametro' en hoja {sheet_name}")



    header_row = [normalize_text(v) for v in raw.iloc[header_idx].tolist()]

    param_col = header_row.index("Parametro")

    llave_col = param_col + 1

    region_start = llave_col + 1



    rows = []

    for i in range(header_idx + 1, len(raw)):

        row = raw.iloc[i]

        parametro = normalize_text(row.iloc[param_col])

        llave = normalize_text(row.iloc[llave_col])

        if parametro is None and llave is None:

            break

        if parametro is None or llave is None:

            continue

        entry = {"Parametro": parametro, "Llave": llave}

        for j, region in enumerate(REGIONES):

            val = safe_float(row.iloc[region_start + j])

            entry[region] = val if val is not None else 0.0

        rows.append(entry)

    return pd.DataFrame(rows)





def filter_sector(df: pd.DataFrame, sector_filter: Callable) -> pd.DataFrame:

    mask = df.apply(lambda r: sector_filter(r["TECHNOLOGY"], r["FUEL"]), axis=1)

    return df[mask].copy()





print(f"Escenario: {ESCENARIO}")

print(f"Nacional: {NACIONAL_FILE.name}")

print(f"Sectores: {', '.join(SECTORES_A_EJECUTAR)}")

print(f"Configuracion: {CONFIG_FILE.name}")

print(f"Regional (inventario): {REGIONAL_FILE.name}")

print(f"Parametros por sector: {len(PARAMS_CONFIG)}")

print(f"Horizonte comparacion: hasta {ANIO_MAX}")



Escenario: BASE
Nacional: 01-04-2026 SAND BASE v10 - copia.xlsx
Sectores: IND, TRA, RES, TER, AGF
Configuracion: Configuracion.xlsx
Regional (inventario): Escenario_SAND_Regional.xlsx
Parametros por sector: 8
Horizonte comparacion: hasta 2054


## 2. Carga de datos



Se cargan Nacional y Regional **una vez**. Luego se procesa cada sector con su filtro y hojas `Cap_*` / `Dem_*`.



In [2]:
df_nac = load_parameters(NACIONAL_FILE)

df_reg = load_parameters(REGIONAL_FILE)

value_cols = get_value_cols(df_nac)

anos = filtrar_anos_2054(value_cols)

compare_cols = ([TIME_INDEP_COL] if TIME_INDEP_COL in value_cols else []) + anos



resumen_sectores = []

for sector, cfg in SECTORS_CONFIG.items():

    sf = cfg["filter"]

    resumen_sectores.append({

        "Sector": sector,

        "Hoja_Cap": cfg["hoja_cap"],

        "Hoja_Dem": cfg["hoja_dem"],

        "Filas_Nacional": len(filter_sector(df_nac, sf)),

        "Filas_Regional": len(filter_sector(df_reg, sf)),

        "Pct_Cap": len(parse_pct_block(cfg["hoja_cap"])),

        "Pct_Dem": len(parse_pct_block(cfg["hoja_dem"])),

    })



show_df(pd.DataFrame(resumen_sectores), "Inventario por sector")

display(Markdown(f"Anos a procesar: **{anos[0]}** a **{anos[-1]}** ({len(anos)} anos)"))



**Inventario por sector**

Sector,Hoja_Cap,Hoja_Dem,Filas_Nacional,Filas_Regional,Pct_Cap,Pct_Dem
IND,Cap_Ind,Dem_Ind,11627,4812,68,7
TRA,Cap_TRA,Dem_TRA,10461,5546,64,13
RES,Cap_RES,Dem_RES,21389,8661,115,16
TER,Cap_TER,Dem_TER,8604,3814,44,10
AGF,Cap_AGF,Dem_AGF,2089,641,5,2


Anos a procesar: **2022** a **2054** (33 anos)

## 3. Generacion regional-first por sector



Por cada sector en `SECTORES_A_EJECUTAR`: filtrar filas, leer % de Config, iterar Regional existente y exportar Excel.



In [3]:


def _mode_base(val):

    return normalize_text(val)





def _tech_base(val):

    _, base = split_regional_name(val)

    return base





def _fuel_base(val):

    _, base = split_regional_name(val)

    return base





def _region_from_row(row: pd.Series) -> str | None:

    return _region_from_name(row["TECHNOLOGY"]) or _region_from_name(row["FUEL"])





def _region_from_name(val):

    region, _ = split_regional_name(val)

    return region





def build_nacional_key(row: pd.Series, dim: str) -> tuple:

    param = normalize_text(row["Parameter"])

    if dim == "fuel":

        return (param, normalize_text(row["FUEL"]))

    if dim == "tech":

        return (param, normalize_text(row["TECHNOLOGY"]))

    return (param, normalize_text(row["TECHNOLOGY"]), _mode_base(row["MODE_OF_OPERATION"]))





def build_regional_key(row: pd.Series, dim: str) -> tuple:

    param = normalize_text(row["Parameter"])

    if dim == "fuel":

        return (param, _fuel_base(row["FUEL"]))

    if dim == "tech":

        return (param, _tech_base(row["TECHNOLOGY"]))

    return (param, _tech_base(row["TECHNOLOGY"]), _mode_base(row["MODE_OF_OPERATION"]))





def lookup_pct(parametro: str, llave: str, pct_df: pd.DataFrame) -> dict[str, float] | None:

    if pct_df is None or pct_df.empty or llave is None:

        return None

    match = pct_df[(pct_df["Parametro"] == parametro) & (pct_df["Llave"] == llave)]

    if match.empty:

        return None

    row = match.iloc[0]

    return {r: safe_float(row[r]) or 0.0 for r in REGIONES}





def get_lookup_key(row: pd.Series, dim: str) -> str | None:

    if dim == "fuel":

        return normalize_text(row["FUEL"])

    if dim in ("tech", "tech_fuel_mode"):

        return normalize_text(row["TECHNOLOGY"])

    return None





def compute_expected_values(nac_row, tipo_efectivo, pct, cols):

    mult = pct if tipo_efectivo == "aditivo" and pct is not None else 1.0

    expected = {}

    for col in cols:

        val = safe_float(nac_row[col])

        expected[col] = round3(val * mult) if val is not None else None

    return expected





def build_recalculated_row(nac_row, reg_row, tipo_efectivo, pct, cols):

    new_row = reg_row.copy()

    mult = pct if tipo_efectivo == "aditivo" and pct is not None else 1.0

    for col in value_cols:

        val = safe_float(nac_row[col])

        if val is None:

            new_row[col] = np.nan

        elif tipo_efectivo == "aditivo":

            new_row[col] = val * mult

        else:

            new_row[col] = val

    return new_row





def find_nacional_row(nac_index, key, reg_row, dim):

    matches = nac_index.get(key, [])

    if not matches:

        return None, False

    if dim != "tech_fuel_mode" or len(matches) == 1:

        return matches[0], False

    reg_fuel_base = _fuel_base(reg_row["FUEL"])

    fuel_matches = [m for m in matches if normalize_text(m["FUEL"]) == reg_fuel_base]

    if fuel_matches:

        return fuel_matches[0], False

    return matches[0], True





def _resumir_discrepancias(disc_rows, total_cols):

    if not disc_rows:

        return pd.DataFrame()

    df = pd.DataFrame(disc_rows)

    group_cols = [

        c for c in df.columns

        if c not in ("Ano", "Valor_Existente", "Valor_Esperado", "Valor_Recalculado", "Diferencia")

    ]

    summary = []

    for key_vals, grp in df.groupby(group_cols, dropna=False):

        if not isinstance(key_vals, tuple):

            key_vals = (key_vals,)

        n_cols = grp["Ano"].nunique()

        max_diff = grp["Diferencia"].max()

        entry = dict(zip(group_cols, key_vals))

        entry["Columnas_Con_Diferencia"] = "todas" if n_cols >= total_cols else n_cols

        entry["Max_Diferencia"] = max_diff

        summary.append(entry)

    return pd.DataFrame(summary)





def export_sector_outputs(sector: str, result: dict) -> tuple[Path, Path]:

    sand_output = OUTPUT_DIR / f"{OUTPUT_PREFIX}_sand_regionalizado_{sector}.xlsx"

    diff_output = OUTPUT_DIR / f"{OUTPUT_PREFIX}_diferencias_regionalizacion_{sector}.xlsx"



    with pd.ExcelWriter(sand_output, engine="openpyxl") as writer:

        result["df_sand"].to_excel(writer, sheet_name="SAND_Recalculado", index=False)

        result["df_resumen_gen"].to_excel(writer, sheet_name="Resumen_Generacion", index=False)

        if not result["df_sin_pct"].empty:

            result["df_sin_pct"].to_excel(writer, sheet_name="Sin_Porcentaje", index=False)



    with pd.ExcelWriter(diff_output, engine="openpyxl") as writer:

        for sheet, key in [

            ("Resumen_Diferencias", "df_disc_resumen"),

            ("Detalle_Diferencias", "df_disc_detalle"),

            ("FUELs_Diferentes", "df_fuels_diff"),

            ("Solo_Regional", "df_solo_regional"),

            ("Con_Pct_Sin_Regional", "df_con_pct_sin_regional"),

            ("Pendientes_Aditivos_Sin_Pct", "df_pendientes_aditivos"),

            ("Sin_Porcentaje", "df_sin_pct"),

            ("Ya_Correctos", "df_ya_correctos"),

            ("Omitidos_Nacional_Cero", "df_omitidos_cero"),

        ]:

            df = result[key]

            if not df.empty:

                df.to_excel(writer, sheet_name=sheet, index=False)

    return sand_output, diff_output





def run_sector(sector: str, sector_cfg: dict) -> dict:

    sector_filter = sector_cfg["filter"]

    hoja_cap = sector_cfg["hoja_cap"]

    hoja_dem = sector_cfg["hoja_dem"]



    nac_sector = filter_sector(df_nac, sector_filter)

    reg_sector = filter_sector(df_reg, sector_filter)

    pct_cap = parse_pct_block(hoja_cap)

    pct_dem = parse_pct_block(hoja_dem)



    sand_rows = []

    omitidos_cero_rows = []

    sin_pct_rows = []

    pendientes_aditivos_rows = []

    solo_regional_rows = []

    ya_correctos_rows = []

    fuels_diff_rows = []

    disc_detalle_rows = []

    resumen_gen = []



    for param, cfg in PARAMS_CONFIG.items():

        dim = cfg["dim"]

        pct_src = cfg.get("pct_source")

        pct_df = pct_dem if pct_src == "dem" else (pct_cap if pct_src == "cap" else None)



        nac_p = nac_sector[nac_sector["Parameter"] == param].copy()

        reg_p = reg_sector[reg_sector["Parameter"] == param].copy()



        nac_index = defaultdict(list)

        for _, nrow in nac_p.iterrows():

            nac_index[build_nacional_key(nrow, dim)].append(nrow)



        n_ok = n_recalc = n_omitidos_cero = n_sin_pct = n_pendientes = n_solo_reg = n_fuels = n_disc = 0



        for _, reg_row in reg_p.iterrows():

            region = _region_from_row(reg_row)

            key = build_regional_key(reg_row, dim)

            nac_row, fuel_ambiguous = find_nacional_row(nac_index, key, reg_row, dim)



            if nac_row is None:

                solo_regional_rows.append({

                    "Parameter": param,

                    "Region": region,

                    "TECHNOLOGY": normalize_text(reg_row["TECHNOLOGY"]),

                    "FUEL": normalize_text(reg_row["FUEL"]),

                    "MODE": _mode_base(reg_row["MODE_OF_OPERATION"]),

                    "Match_Key": str(key),

                    "Motivo": "existe en Regional, no en Nacional",

                })

                n_solo_reg += 1

                continue



            if dim == "tech_fuel_mode":

                nac_fuel_base = normalize_text(nac_row["FUEL"])

                reg_fuel_base = _fuel_base(reg_row["FUEL"])

                if nac_fuel_base != reg_fuel_base or fuel_ambiguous:

                    fuels_diff_rows.append({

                        "Parameter": param,

                        "Region": region,

                        "TECHNOLOGY_Base": key[2] if len(key) > 2 else key[1],

                        "MODE": key[3] if len(key) > 3 else None,

                        "FUEL_Nacional": nac_fuel_base,

                        "FUEL_Regional": normalize_text(reg_row["FUEL"]),

                        "Ambiguedad_Nacional": fuel_ambiguous,

                    })

                    n_fuels += 1



            tipo_efectivo = get_tipo_efectivo(param, nac_row, compare_cols)

            llave = get_lookup_key(nac_row, dim)

            pct = None

            if tipo_efectivo == "aditivo":

                pcts = lookup_pct(param, llave, pct_df)

                if pcts is None:

                    entry = {

                        "Parameter": param,

                        "Region": region,

                        "Llave": llave,

                        "TECHNOLOGY": normalize_text(reg_row["TECHNOLOGY"]),

                        "FUEL": normalize_text(reg_row["FUEL"]),

                        "Motivo": "aditivo sin porcentaje en configuracion",

                    }

                    if param in PARAMS_CONDICIONALES:

                        pendientes_aditivos_rows.append(entry)

                        n_pendientes += 1

                    else:

                        sin_pct_rows.append(entry)

                        n_sin_pct += 1

                    continue

                pct = pcts.get(region, 0.0) if region else 0.0

                if pct is None or pct <= 0:

                    continue



            expected = compute_expected_values(nac_row, tipo_efectivo, pct, compare_cols)

            tiene_disc = False

            for col in compare_cols:

                v_reg = safe_float(reg_row[col])

                v_esp = expected[col]

                if not values_match_3dec(v_reg, v_esp):

                    tiene_disc = True

                    diff = None

                    if v_reg is not None and v_esp is not None:

                        diff = abs(round3(v_reg) - round3(v_esp))

                    disc_detalle_rows.append({

                        "Parameter": param,

                        "Region": region,

                        "TECHNOLOGY": normalize_text(reg_row["TECHNOLOGY"]),

                        "FUEL": normalize_text(reg_row["FUEL"]),

                        "MODE": _mode_base(reg_row["MODE_OF_OPERATION"]),

                        "Tipo_Efectivo": tipo_efectivo,

                        "Ano": col,

                        "Valor_Existente": v_reg,

                        "Valor_Esperado": v_esp,

                        "Diferencia": diff,

                    })

                    n_disc += 1



            if tiene_disc:

                if nacional_es_todo_cero(nac_row, compare_cols):

                    omitidos_cero_rows.append({

                        "Parameter": param,

                        "Region": region,

                        "TECHNOLOGY": normalize_text(reg_row["TECHNOLOGY"]),

                        "FUEL": normalize_text(reg_row["FUEL"]),

                        "MODE": _mode_base(reg_row["MODE_OF_OPERATION"]),

                        "Tipo_Efectivo": tipo_efectivo,

                        "Motivo": "Nacional todo cero; no exportar a SAND",

                    })

                    n_omitidos_cero += 1

                else:

                    sand_rows.append(build_recalculated_row(nac_row, reg_row, tipo_efectivo, pct, compare_cols))

                    n_recalc += 1

            else:

                ya_correctos_rows.append({

                    "Parameter": param,

                    "Region": region,

                    "TECHNOLOGY": normalize_text(reg_row["TECHNOLOGY"]),

                    "FUEL": normalize_text(reg_row["FUEL"]),

                    "MODE": _mode_base(reg_row["MODE_OF_OPERATION"]),

                    "Tipo_Efectivo": tipo_efectivo,

                })

                n_ok += 1



        resumen_gen.append({

            "Parameter": param,

            "Filas_Regional": len(reg_p),

            "Ya_Correctas": n_ok,

            "Recalculadas": n_recalc,

            "Omitidos_Nacional_Cero": n_omitidos_cero,

            "Sin_Porcentaje": n_sin_pct,

            "Pendientes_Aditivos": n_pendientes,

            "Solo_Regional": n_solo_reg,

            "FUELs_Diferentes": n_fuels,

            "Discrepancias_Detalle": n_disc,

        })



    regional_llaves = set()

    for param, cfg in PARAMS_CONFIG.items():

        if cfg.get("pct_source") is None:

            continue

        dim = cfg["dim"]

        reg_p = reg_sector[reg_sector["Parameter"] == param]

        for _, row in reg_p.iterrows():

            llave = _fuel_base(row["FUEL"]) if dim == "fuel" else _tech_base(row["TECHNOLOGY"])

            if llave:

                regional_llaves.add((param, llave))



    con_pct_sin_regional_rows = []

    for pct_df, _hoja in [(pct_cap, hoja_cap), (pct_dem, hoja_dem)]:

        for _, row in pct_df.iterrows():

            param = row["Parametro"]

            llave = row["Llave"]

            key = (param, llave)

            if key in regional_llaves:

                continue

            if not any((safe_float(row[r]) or 0.0) > 0 for r in REGIONES):

                continue

            con_pct_sin_regional_rows.append({

                "Parametro": param,

                "Llave": llave,

                "Motivo": f"pct en Configuracion pero sin fila Regional {sector}",

                **{r: safe_float(row[r]) for r in REGIONES},

            })



    return {

        "sector": sector,

        "hoja_cap": hoja_cap,

        "hoja_dem": hoja_dem,

        "df_sand": pd.DataFrame(sand_rows).reset_index(drop=True) if sand_rows else pd.DataFrame(columns=df_nac.columns),

        "df_omitidos_cero": pd.DataFrame(omitidos_cero_rows),

        "df_sin_pct": pd.DataFrame(sin_pct_rows),

        "df_pendientes_aditivos": pd.DataFrame(pendientes_aditivos_rows),

        "df_solo_regional": pd.DataFrame(solo_regional_rows),

        "df_ya_correctos": pd.DataFrame(ya_correctos_rows),

        "df_fuels_diff": pd.DataFrame(fuels_diff_rows),

        "df_disc_detalle": pd.DataFrame(disc_detalle_rows),

        "df_disc_resumen": _resumir_discrepancias(disc_detalle_rows, len(compare_cols)),

        "df_con_pct_sin_regional": pd.DataFrame(con_pct_sin_regional_rows),

        "df_resumen_gen": pd.DataFrame(resumen_gen),

    }





resultados_por_sector: dict[str, dict] = {}

export_paths: list[tuple[str, Path, Path]] = []



for sector in SECTORES_A_EJECUTAR:

    display(Markdown(f"---\n## Sector **{sector}** ({SECTORS_CONFIG[sector]['hoja_cap']}, {SECTORS_CONFIG[sector]['hoja_dem']})"))

    result = run_sector(sector, SECTORS_CONFIG[sector])

    resultados_por_sector[sector] = result

    sand_path, diff_path = export_sector_outputs(sector, result)

    export_paths.append((sector, sand_path, diff_path))



    display(Markdown(

        f"### {sector}: SAND recalculado **{len(result['df_sand']):,}** filas | "

        f"Pendientes aditivos **{len(result['df_pendientes_aditivos'])}** | "

        f"Omitidos cero **{len(result['df_omitidos_cero'])}**"

    ))

    show_df(result["df_resumen_gen"], f"Resumen generacion - {sector}")



resumen_todos = []

for sector, result in resultados_por_sector.items():

    rg = result["df_resumen_gen"]

    if rg.empty:

        continue

    tmp = rg.copy()

    tmp.insert(0, "Sector", sector)

    resumen_todos.append(tmp)



df_resumen_todos_sectores = pd.concat(resumen_todos, ignore_index=True) if resumen_todos else pd.DataFrame()

show_df(df_resumen_todos_sectores, "Resumen consolidado - todos los sectores")



---
## Sector **IND** (Cap_Ind, Dem_Ind)

### IND: SAND recalculado **671** filas | Pendientes aditivos **0** | Omitidos cero **10**

**Resumen generacion - IND**

Parameter,Filas_Regional,Ya_Correctas,Recalculadas,Omitidos_Nacional_Cero,Sin_Porcentaje,Pendientes_Aditivos,Solo_Regional,FUELs_Diferentes,Discrepancias_Detalle
AccumulatedAnnualDemand,30,0,30,0,0,0,0,0,990
ResidualCapacity,300,0,179,0,0,0,0,0,5769
TotalAnnualMaxCapacityInvestment,14,14,0,0,0,0,0,0,0
TotalTechnologyAnnualActivityUpperLimit,10,10,0,0,0,0,0,0,0
TotalTechnologyModelPeriodActivityUpperLimit,369,0,329,0,0,0,40,0,329
CapitalCost,359,277,32,10,0,0,40,0,1386
InputActivityRatio,319,218,101,0,0,0,0,115,3222
OutputActivityRatio,369,329,0,0,0,0,40,0,0


---
## Sector **TRA** (Cap_TRA, Dem_TRA)

...

### AGF: SAND recalculado **57** filas | Pendientes aditivos **0** | Omitidos cero **0**

**Resumen generacion - AGF**

Parameter,Filas_Regional,Ya_Correctas,Recalculadas,Omitidos_Nacional_Cero,Sin_Porcentaje,Pendientes_Aditivos,Solo_Regional,FUELs_Diferentes,Discrepancias_Detalle
AccumulatedAnnualDemand,14,14,0,0,0,0,0,0,0
ResidualCapacity,39,28,5,0,6,0,0,0,37
TotalAnnualMaxCapacityInvestment,0,0,0,0,0,0,0,0,0
TotalTechnologyAnnualActivityUpperLimit,0,0,0,0,0,0,0,0,0
TotalTechnologyModelPeriodActivityUpperLimit,55,0,41,0,0,0,14,0,41
CapitalCost,53,32,7,0,0,0,14,0,231
InputActivityRatio,41,37,4,0,0,0,0,21,128
OutputActivityRatio,55,41,0,0,0,0,14,0,0


**Resumen consolidado - todos los sectores**

Sector,Parameter,Filas_Regional,Ya_Correctas,Recalculadas,Omitidos_Nacional_Cero,Sin_Porcentaje,Pendientes_Aditivos,Solo_Regional,FUELs_Diferentes,Discrepancias_Detalle
IND,AccumulatedAnnualDemand,30,0,30,0,0,0,0,0,990
IND,ResidualCapacity,300,0,179,0,0,0,0,0,5769
IND,TotalAnnualMaxCapacityInvestment,14,14,0,0,0,0,0,0,0
IND,TotalTechnologyAnnualActivityUpperLimit,10,10,0,0,0,0,0,0,0
IND,TotalTechnologyModelPeriodActivityUpperLimit,369,0,329,0,0,0,40,0,329
IND,CapitalCost,359,277,32,10,0,0,40,0,1386
IND,InputActivityRatio,319,218,101,0,0,0,0,115,3222
IND,OutputActivityRatio,369,329,0,0,0,0,40,0,0
TRA,AccumulatedAnnualDemand,58,40,18,0,0,0,0,0,516
TRA,ResidualCapacity,361,204,44,0,0,0,5,0,382


## 4. Resumen global y archivos exportados



In [4]:
display(Markdown("### Resumen global por sector"))



filas_resumen = []

for sector, result in resultados_por_sector.items():

    filas_resumen.append({

        "Sector": sector,

        "Recalculadas": len(result["df_sand"]),

        "Ya_Correctas": len(result["df_ya_correctos"]),

        "Omitidos_Nacional_Cero": len(result["df_omitidos_cero"]),

        "Discrepancias": len(result["df_disc_detalle"]),

        "Pendientes_Aditivos": len(result["df_pendientes_aditivos"]),

        "Solo_Regional": len(result["df_solo_regional"]),

        "Con_Pct_Sin_Regional": len(result["df_con_pct_sin_regional"]),

    })



show_df(pd.DataFrame(filas_resumen), "Totales por sector")



lines = ["### Archivos exportados"]

for sector, sand_path, diff_path in export_paths:

    lines.append(f"- **{sector}**: `{sand_path.name}`, `{diff_path.name}`")

display(Markdown("\n".join(lines)))



### Resumen global por sector

**Totales por sector**

Sector,Recalculadas,Ya_Correctas,Omitidos_Nacional_Cero,Discrepancias,Pendientes_Aditivos,Solo_Regional,Con_Pct_Sin_Regional
IND,671,848,10,11696,0,120,2
TRA,425,1283,34,2577,0,296,1
RES,1158,1475,44,19861,0,622,0
TER,433,782,35,5419,0,212,0
AGF,57,152,0,437,0,42,0


### Archivos exportados
- **IND**: `BASE_sand_regionalizado_IND.xlsx`, `BASE_diferencias_regionalizacion_IND.xlsx`
- **TRA**: `BASE_sand_regionalizado_TRA.xlsx`, `BASE_diferencias_regionalizacion_TRA.xlsx`
- **RES**: `BASE_sand_regionalizado_RES.xlsx`, `BASE_diferencias_regionalizacion_RES.xlsx`
- **TER**: `BASE_sand_regionalizado_TER.xlsx`, `BASE_diferencias_regionalizacion_TER.xlsx`
- **AGF**: `BASE_sand_regionalizado_AGF.xlsx`, `BASE_diferencias_regionalizacion_AGF.xlsx`